# WEEK 2: RAG PIPELINE WITH BM25 + PAGEINDEX + GEMINI

In [ ]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25

In [50]:
# ── SECTION 0: Set up API Keys ─────────────────────────────

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
PAGEINDEX_API_KEY = userdata.get("PAGEINDEX")
print("GOOGLE_API_KEY found:", os.environ["GOOGLE_API_KEY"] is not None)
print("PAGEINDEX found:", PAGEINDEX_API_KEY is not None)

GOOGLE_API_KEY found: True
PAGEINDEX found: True


## Corpus Preparation

In [ ]:
# ── SECTION 1.1: Load the raw CSV data ──────────────────────
import pandas as pd

df = pd.read_csv('data.csv')
df

,url,title,text,tokens,token_count
0,https://www.bbc.com/news/articles/crlp991nw41o,Why the AI industry is the real winner of the ...,It is not only OpenAI but the AI race itself t...,"['it', 'is', 'not', 'only', 'openai', 'but', '...",946
1,https://www.bbc.com/future/article/20260519-go...,Google's AI is being manipulated. The search g...,A BBC investigation revealed a simple way AI c...,"['a', 'bbc', 'investigation', 'revealed', 'a',...",1860
2,https://www.bbc.com/news/articles/cvgz1ynq1nqo,Google to release first smart glasses since Go...,More than a decade after its famed Google Glas...,"['more', 'than', 'a', 'decade', 'after', 'its'...",672
3,https://www.bbc.com/news/articles/crep3v8vzglo,Standard Chartered to cut thousands of roles a...,Banking giant Standard Chartered has become th...,"['banking', 'giant', 'standard', 'chartered', ...",463
4,https://www.bbc.com/news/articles/c4g04qkqlk2o,Samsung strike on hold as workers push for AI ...,The largest union at Samsung Electronics has s...,"['the', 'largest', 'union', 'at', 'samsung', '...",789
5,https://www.bbc.com/future/article/20260505-ho...,'Think outside the bots': How to stop AI from ...,GPS ruined our sense of direction. Search engi...,"['gps', 'ruined', 'our', 'sense', 'of', 'direc...",2153
6,https://www.bbc.com/news/articles/c0q2gkj97eko,Robo-top: The machines that could make your ne...,"They assemble cars, perform surgery and even h...","['they', 'assemble', 'cars', 'perform', 'surge...",1187
7,https://www.bbc.com/news/articles/c62jp09p0l4o,The fight against foreign developers buying Ca...,"On the small Caribbean island of Barbuda, the ...","['on', 'the', 'small', 'caribbean', 'island', ...",1346
8,https://www.bbc.com/future/article/20251218-ho...,"They hear, but do they care? What AI can teach...",AI chatbots don't interrupt and aren't judgeme...,"['ai', 'chatbots', 'don', 't', 'interrupt', 'a...",2335
9,https://www.bbc.com/news/articles/cvgzr935jvmo,University of Lincoln AI stand wins award at C...,An exhibition featuring a health scan for plan...,"['an', 'exhibition', 'featuring', 'a', 'health...",321


In [ ]:
# Checking datatype of token
print(type(df["tokens"].iloc[0]))
print(df["tokens"].iloc[0][:20])

<class 'str'>
['it', 'is', 'not', 


In [ ]:
# ── SECTION 1.2: Convert token strings back to actual lists ──
# When pandas reads a CSV, everything comes in as a string
# ast.literal_eval() safely converts that string BACK into a real list.
import ast
df["tokens"] = df["tokens"].apply(ast.literal_eval)

# BM25 and text splitters work on strings
df["clean_text"] = [" ".join(x) for x in df["tokens"]] # Join the token list back into a single string sentence.
df

# Keep only the columns we'll actually use downstream
df = df[["url", "title", "clean_text"]]

In [ ]:
!pip install langchain-text-splitters

In [ ]:
# ── SECTION 1.3: Split articles into smaller chunks ──────────
# Gemini has a context window limit — you can't dump 10 full articles.

from langchain_text_splitters import RecursiveCharacterTextSplitter as rts

# So we break each article into overlapping pieces of ~1000 characters.
# chunk_overlap=200 means consecutive chunks share 200 characters.
txt_spliter = rts(chunk_size= 1000, chunk_overlap=200)

all_chunks = []

for i in range(len(df)):
    text = df.loc[i, "clean_text"]
    title = df.loc[i, "title"]
    url = df.loc[i, "url"]

    pieces = txt_spliter.split_text(text) # split_text returns a list of string pieces

    # Store each chunk with metadata so we know which article it came from
    for j in range(len(pieces)):
        all_chunks.append({
            "article_id": i, # which article this chunk belongs to
            "chunk_id": j, # which piece within that article
            "title": title,
            "url": url,
            "text": pieces[j]
        })

print("Total chunks made:", len(all_chunks))
print(all_chunks[0])
print(all_chunks[1])

Total chunks made: 91
{'article_id': 0, 'chunk_id': 0, 'title': 'Why the AI industry is the real winner of the Musk-Altman trial', 'url': 'https://www.bbc.com/news/articles/crlp991nw41o', 'text': 'it is not only openai but the ai race itself that was vindicated in the california courtroom last night even though elon musk essentially lost on a technicality there s a clear signal from the verdict that making lots of money from ai and competing fiercely with rivals is simply business the industry sometimes tries to display a united front especially when it comes to safety research and inclusivity but this case served as a powerful reminder that none of the ai giants are charities and don t have to be even if they once said otherwise cracks in the façade of industry collaboration for the sake of humanity have been exposed before in february i was in india for a global ai summit where host prime minister narendra modi orchestrated the world s tech leaders to hold hands on stage sam altman a

In [ ]:
# New datafarame
df = pd.DataFrame(all_chunks)
df

,article_id,chunk_id,title,url,text
0,0,0,Why the AI industry is the real winner of the ...,https://www.bbc.com/news/articles/crlp991nw41o,it is not only openai but the ai race itself t...
1,0,1,Why the AI industry is the real winner of the ...,https://www.bbc.com/news/articles/crlp991nw41o,and anthropic boss dario amodei once colleague...
2,0,2,Why the AI industry is the real winner of the ...,https://www.bbc.com/news/articles/crlp991nw41o,its own money i met dresser who joined openai ...
3,0,3,Why the AI industry is the real winner of the ...,https://www.bbc.com/news/articles/crlp991nw41o,is not his first rodeo in courtroom dramas and...
4,0,4,Why the AI industry is the real winner of the ...,https://www.bbc.com/news/articles/crlp991nw41o,value in ai but it also exposed some of the im...
...,...,...,...,...,...
86,8,16,"They hear, but do they care? What AI can teach...",https://www.bbc.com/future/article/20251218-ho...,flying fears persist despite falling accident ...
87,8,17,"They hear, but do they care? What AI can teach...",https://www.bbc.com/future/article/20251218-ho...,at first overlooked whistler s 1871 painting o...
88,9,0,University of Lincoln AI stand wins award at C...,https://www.bbc.com/news/articles/cvgzr935jvmo,an exhibition featuring a health scan for plan...
89,9,1,University of Lincoln AI stand wins award at C...,https://www.bbc.com/news/articles/cvgzr935jvmo,said one of the most rewarding aspects we ve a...


In [ ]:
# ── SECTION 1.4: Save chunks to CSV ─────────────────
df.to_csv("chunked_corpus.csv", index=False)
print("Saved chunked_corpus.csv")

Saved chunked_corpus.csv


## Hierarchical PageIndex

In [ ]:
!pip install fpdf

In [ ]:
# ── SECTION 2.1: Save chunks to PDF ─────────────────
# Why PDF? Because PageIndex accepts only PDFs as input.

import pandas as pd
from fpdf import FPDF

df = pd.read_csv("chunked_corpus.csv")

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15) # auto-adds new pages when full

for i in range(len(df)):
    title = str(df.loc[i, "title"])
    text = str(df.loc[i, "text"])
    url = str(df.loc[i, "url"])

    pdf.add_page()

    # Bold, larger font for the title
    pdf.set_font("Arial", "B", 14)
    pdf.multi_cell(0, 10, f"Title: {title}")

    # Smaller font for the URL
    pdf.set_font("Arial", "", 10)
    pdf.multi_cell(0, 8, f"URL: {url}")

    pdf.ln(2)  # small vertical gap

    # Regular font for the chunk text
    pdf.set_font("Arial", "", 12)
    pdf.multi_cell(0, 8, text)

pdf.output("chunked_corpus.pdf")
print("PDF created: chunked_corpus.pdf")

PDF created: chunked_corpus.pdf


In [ ]:
# ── SECTION 2.2: Index the PDF with PageIndex ────────────────
from pageindex import PageIndexClient
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

import pageindex.utils as utils

doc = pi_client.submit_document("chunked_corpus.pdf") # Submit the PDF to PageIndex for processing.
doc_id = doc["doc_id"]

while not pi_client.is_retrieval_ready(doc_id):
    print("Still indexing...")
    import time
    time.sleep(5)

tree = pi_client.get_tree(doc_id, node_summary=True)["result"] # Fetch the full document tree with node summaries

print("Document Tree:")
utils.print_tree(tree) # prints the hierarchy

Still indexing...
Still indexing...
Still indexing...
Document Tree:
[{'title': 'Title: Why the AI industry is the real w...',
  'node_id': '0000',
  'summary': 'The Musk-Altman trial verdict confirmed ...'},
 {'title': "Title: Google's AI is being manipulated....",
  'node_id': '0001',
  'prefix_summary': 'A BBC investigation reveals that AI chat...',
  'nodes': [{'title': 'The Vulnerability of AI to Online Manipu...',
             'node_id': '0002',
             'summary': 'A BBC investigation highlights how AI ch...'},
            {'title': 'The Ongoing Battle Against AI Search Man...',
             'node_id': '0003',
             'summary': 'The text explores the systemic manipulat...'},
            {'title': 'The Evolving AI Landscape and Global Tec...',
             'node_id': '0004',
             'summary': 'The text discusses the ongoing struggle ...'},
            {'title': 'Technological Innovations and Diverse Me...',
             'node_id': '0005',
             'summary': '

## BM25 Retrieval — Implementation

In [ ]:
# ── SECTION 3.1: Flatten the tree into a list of nodes ───────
# BM25 needs a flat list of text chunks to score.
# So we walk the tree recursively and collect every node's summary text.

documents = [] # will hold one dict per tree node

""" Recursively walks through the PageIndex tree.

    Each node has:
    - node_id: unique ID
    - title: heading/label for this section
    - summary: a short auto-generated description of this section
    - nodes: list of child nodes (may be empty for leaf nodes)

    We prefer 'summary' over 'prefix_summary' because summary is
    usually more complete. prefix_summary is a fallback."""

def traverse_tree(node, parent_id=None, level=0):
    node_id = node.get("node_id", "")
    title = node.get("title", "")
    summary = node.get("summary", "")
    prefix_summary = node.get("prefix_summary", "")

    text = summary if summary else prefix_summary

    documents.append({
        "node_id": node_id,
        "parent_id": parent_id, # track the hierarchy
        "level": level, # depth in the tree — root=0, children=1, etc.
        "title": title,
        "text": text
    })

    # Recurse into children if any exist
    if "nodes" in node:
        for child in node["nodes"]:
            traverse_tree(child, parent_id=node_id, level=level + 1)

# Start traversal from every top-level node
for top_node in tree:
    traverse_tree(top_node)

In [ ]:
# ── SECTION 3.2: Build the BM25 Index ───────────────────────
from rank_bm25 import BM25Okapi

# convert each node text into tokens
tokenized_corpus = []

for doc in documents:
    tokens = doc["text"].split()
    tokenized_corpus.append(tokens)

# build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built")
print("Number of documents in BM25:", len(tokenized_corpus))

BM25 index built
Number of documents in BM25: 29


In [ ]:
# ── SECTION 3.3: Test a raw BM25 query ──────────────────────
query = "AI manipulation of search results"
query_tokens = query.lower().split()

scores = bm25.get_scores(query_tokens)
print(scores)

# get indices of top 3 matching docs
top_n = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]

for i in top_n:
    print("Score:", scores[i])
    print("Title:", documents[i]["title"])
    print("Text:", documents[i]["text"])
    print("-" * 50)

[1.0364215  2.80594289 2.24713515 7.87352831 0.94544208 0.99734519
 0.         1.06041113 0.89498459 2.23908872 2.31543696 0.90614464
 0.95371416 1.11410441 0.79185511 0.64249707 1.16130991 0.65406281
 1.1203564  0.88409609 1.14906782 1.01593623 0.7332604  0.63133325
 0.         1.14604752 1.30461642 1.25129563 0.7002848 ]
Score: 7.873528306061369
Title: The Ongoing Battle Against AI Search Manipulation
Text: The text explores the systemic manipulation of AI search results and the potential dangers of biased or inaccurate information in critical areas like health and law. It details Google's efforts to combat this through updated spam policies and silent algorithmic adjustments, such as filtering out self-promotional content and adding transparency labels. Despite these interventions, experts remain skeptical, noting that manipulators continuously evolve their tactics—such as leveraging social media influencers—to circumvent new rules, leaving AI companies in a persistent, reactive str

In [ ]:
# ── SECTION 3.4: Wrap BM25 as a LangChain Retriever ────────

from typing import List
from pydantic import Field
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

# LangChain's retrieval chain expects a retriever object that follows its interface (a class with _get_relevant_documents method).
# We wrap our BM25 logic in that interface so it plugs in cleanly.
class BM25Retriever(BaseRetriever):
    bm25: object = Field()  # our BM25 index object
    docs: list   = Field()  # our flat list of node dicts
    k:    int    = 3        # how many top results to return

    def _get_relevant_documents(self, query: str) -> List[Document]:
        # convert query into tokens
        query_tokens = query.lower().split()

        # score all documents
        scores = self.bm25.get_scores(query_tokens)

        # get top k indices
        top_n = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:self.k]

        # convert results into LangChain Document objects
        results = []
        for i in top_n:
            results.append(
                Document(
                    page_content=self.docs[i]["text"],
                    metadata={
                        "node_id": self.docs[i]["node_id"],
                        "title": self.docs[i]["title"],
                        "score": float(scores[i])
                    }
                )
            )
        return results

In [ ]:
# create actual working retriever using our BM25 index
retriever = BM25Retriever(bm25=bm25, docs=documents, k=3)
print("Retriever ready")

Retriever ready


In [ ]:
# Quick test
results = retriever.invoke("AI manipulation of search results")

for doc in results:
    print(doc.metadata)
    print(doc.page_content)
    print("-" * 50)

{'node_id': '0003', 'title': 'The Ongoing Battle Against AI Search Manipulation', 'score': 7.873528306061369}
The text explores the systemic manipulation of AI search results and the potential dangers of biased or inaccurate information in critical areas like health and law. It details Google's efforts to combat this through updated spam policies and silent algorithmic adjustments, such as filtering out self-promotional content and adding transparency labels. Despite these interventions, experts remain skeptical, noting that manipulators continuously evolve their tactics—such as leveraging social media influencers—to circumvent new rules, leaving AI companies in a persistent, reactive struggle to maintain result integrity.
--------------------------------------------------
{'node_id': '0001', 'title': "Title: Google's AI is being manipulated. The search giant is quietly fighting back", 'score': 2.8059428868592873}
A BBC investigation reveals that AI chatbots and search tools are being 

## LangChain Integration

In [ ]:
!pip install -U langchain-classic

In [51]:
# ── SECTION 4.1: Set up Gemini + Prompt ────────
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite") # We're using Gemini as the LLM here.

prompt = ChatPromptTemplate.from_template("""
Answer the question only using the context below.
If the answer is not present in the context, say: "Answer not found in the retrieved context."

Context: {context}

Question: {input}
""")

In [52]:
# ── SECTION 4.2: Run the RAG pipeline ──────────────────────

# takes retrieved docs and "stuffs" them all into the {context} slot of the prompt.
document_chain = create_stuff_documents_chain(llm, prompt)
# create_retrieval_chain: first runs the retriever, then passes the results to the document_chain.
retrieval_chain = create_retrieval_chain(retriever, document_chain)

response = retrieval_chain.invoke({
    "input": "How is AI being manipulated online?"
})

print(response["answer"])

AI is being manipulated by the use of simple, strategically published online content, which can force AI tools to generate biased or false information.


## Latency & Accuracy Comparison

In [57]:
# ── SECTION 5.1: Compare RAG vs Naive Full-Context ──────────
question = "How is AI being manipulated online?"

# naive: dump everything into the prompt
full_context    = " ".join(df["text"].astype(str).tolist())
formatted_prompt = prompt.format(context=full_context, input=question)

start       = time.perf_counter()
naive_answer = llm.invoke(formatted_prompt).content
naive_time  = time.perf_counter() - start

# rag: retriever picks top 3 chunks
start       = time.perf_counter()
rag_response = retrieval_chain.invoke({"input": question})
rag_time    = time.perf_counter() - start
rag_answer  = rag_response["answer"]
rag_chunks  = [doc.page_content for doc in rag_response["context"]]

In [58]:
!pip install deepeval

In [64]:
# ── SECTION 5.2: Accuracy comparison (DeepEval) ─────────────
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRelevancyMetric, FaithfulnessMetric
from deepeval.models import GeminiModel

judge = GeminiModel(model="gemini-3.1-flash-lite")

# build one test case per method
naive_case = LLMTestCase(
    input=question,
    actual_output=naive_answer[0]["text"],
    retrieval_context=[full_context]  # one giant blob
)
rag_case = LLMTestCase(
    input=question,
    actual_output=rag_answer,
    retrieval_context=rag_chunks       # 3 targeted chunks
)

# run metrics on both
relevancy    = ContextualRelevancyMetric(threshold=0.7, model=judge)
faithfulness = FaithfulnessMetric(threshold=0.7, model=judge, include_reason=True)

relevancy.measure(naive_case)
naive_rel   = relevancy.score
faithfulness.measure(naive_case)
naive_faith = faithfulness.score

relevancy.measure(rag_case)
rag_rel     = relevancy.score
faithfulness.measure(rag_case)
rag_faith   = faithfulness.score

# final table
pd.DataFrame([
    {"method": "Naive full-context", "latency_sec": round(naive_time, 2), "relevancy": naive_rel,  "faithfulness": naive_faith},
    {"method": "BM25 RAG",           "latency_sec": round(rag_time, 2),   "relevancy": rag_rel,    "faithfulness": rag_faith}
])

Output()

Output()

Output()

Output()

,method,latency_sec,relevancy,faithfulness
0,Naive full-context,1.50,0.700000,1.0
1,BM25 RAG,0.57,0.666667,1.0
